# L27 · 可观测性：看清 AI 在想什么

**学习目标**
- 理解「可观测性（Observability）」：日志、指标、追踪三件套
- 理解 LLM 应用的「追踪（Tracing）」：一次回答背后调了哪些步骤
- 亲手实现一个「调用追踪器」，把一次 AI 请求的链路画出来

**前置依赖**：L23（Agent 多步）、L26（护栏）  
**预计时长**：50 分钟  
**技术栈**：纯 Python（离线追踪，无需 LLM）

---

## 概念讲解：可观测性 = 给 AI 装「黑匣子」

AI 出错时，你不能问它「你刚才为啥答错」。所以要提前埋点记录：
- **日志（Logs）**：发生了什么（文字记录）
- **指标（Metrics）**：量了多少（延迟、成功率、花费）
- **追踪（Traces）**：一次请求内部走了哪些步骤、每步多久

本课聚焦 **Tracing**——把一次 AI 回答的「内心活动」可视化，这是调试 Agent 的必备技能。

## 第一步：写一个「会记录每一步」的追踪器

In [ ]:
import time

class Tracer:
    def __init__(self):
        self.spans = []          # 每一步记录
    def span(self, name, fn, *args):
        t0 = time.time()
        out = fn(*args)
        cost = (time.time() - t0) * 1000
        self.spans.append((name, cost, out))
        return out

tracer = Tracer()
print("✅ 追踪器就绪")

## 第二步：用追踪器跑一个「多步 AI 任务」

In [ ]:
import random
def retrieve(q): time.sleep(0.1); return f"检索到关于'{q}'的3段资料"
def generate(q, ctx): time.sleep(0.2); return f"基于资料生成回答：{q[:6]}..."
def guard(text): time.sleep(0.05); return text + "（已安检）"

q = "北京天气如何"
ctx = tracer.span("RAG检索", retrieve, q)
ans = tracer.span("LLM生成", generate, q, ctx)
final = tracer.span("护栏检查", guard, ans)
print("最终输出：", final)

# 🎯 AHA 顿悟单元格：把一次 AI 调用「拆开给你看」

运行下面代码。你会看到一次 AI 回答背后的**完整调用链**：检索→生成→安检，
每步耗时、每步产物都列得清清楚楚，还有一张 ASCII 时间轴。改任务看链路如何变化。

> 你刚写的 Tracer，和 LangSmith、Langfuse、OpenTelemetry 是同一类工具。
> 没有它，AI 出错时你只能干瞪眼。有了它，你能「看穿」模型的每一步。

In [ ]:
# ===== 运行我！看一次 AI 调用的内部链路 =====
import time, random
tracer = Tracer()
q = "帮我总结公司 Q3 财报要点"
ctx = tracer.span("RAG检索", retrieve, q)
ans = tracer.span("LLM生成", generate, q, ctx)
final = tracer.span("护栏检查", guard, ans)

print("\n  🔍 一次 AI 调用的内部追踪（Trace）：\n")
print("  " + "=" * 48)
total = sum(s[1] for s in tracer.spans)
for i, (name, cost, out) in enumerate(tracer.spans, 1):
    bar = "█" * int(cost / 2)
    print(f"  {i}. {name:<10} {cost:6.1f}ms {bar}")
    print(f"     产出: {out}")
print("  " + "=" * 48)
print(f"  ⏱️  总耗时 {total:.1f}ms，共 {len(tracer.spans)} 个环节")
print("  ✨ 你拥有了『看穿 AI 内心』的能力！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：Logs/Metrics/Traces 三件套区分；Trace span 的父子嵌套（本课简化为平铺，可在备课笔记提真场景嵌套）。  
**易错点**：`time.sleep` 量级控制（用 0.1s 模拟，避免真实等待）。  
**AHA 机制**：调用链+时间轴可视化，强「可调试」专业感。  
**衔接**：L28 安全对齐（追踪可记录违规）；L30 MLOps（监控指标）。  
**依赖**：纯 Python 标准库。  
**SOTA 工具**：对标 LangSmith/Langfuse/OpenTelemetry，注明本课是手搓最小版。

# 📚 作业 / 下一步

1. 给追踪器加一个「嵌套子步骤」（如 LLM 生成内部再分 思考/写草稿）。
2. 把每步耗时累加成一个「成本估算」。
3. 下一课 **L28 安全与对齐：防止 AI 作恶** —— 从工程护栏走向价值观对齐。